# Sequence Labeling: The Linguistic Detective Agency

**Mission Briefing**

Welcome, **Agent IoB**. You have been recruited by the *Bureau of Computational Linguistics*.

Your Mission: Decode the hidden structures underlying text. We see the words (the "Observations"), but the true meanings—the grammatical roles and named entities—are hidden in the shadows (the "States").

**Objective**:
1.  **Identify Suspects**: Classify words into their Part-of-Speech (POS) roles.
2.  **Trace the Shadow Path**: Use Hidden Markov Models (HMMs) to model the suspect's behavior.
3.  **Find the Optimal Path**: Implement the Viterbi Algorithm to decode the most likely sequence of events.
4.  **Extract Intel**: Use Named Entity Recognition (NER) to find high-value targets (People, Organizations).
5.  **Contextual Forensics**: Use Conditional Random Fields (CRFs) for advanced analysis.

---

In [ ]:
# Mission Gear (Setup)
!pip install nltk spacy sklearn-crfsuite matplotlib pandas numpy
!python -m spacy download en_core_web_sm

import nltk
import spacy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

nltk.download('treebank')
nltk.download('universal_tagset')
nltk.download('brown')

## 1. The Suspects (Word Classes)

Before we can track them, we must know who they are. Words fall into two main categories:

1.  **Open Class** (The Civilians): Neologisms appear daily. (Nouns, Verbs, Adjectives). Infinite variety.
2.  **Closed Class** (The Officials): A fixed, small set. (Prepositions, Determiners, Pronouns). They rarely change.

The **Penn Treebank** is our standard dossier for English tags.

In [ ]:
# Loading the Dossier (Corpus)
from nltk.corpus import brown

# Check the tags used in the Brown Corpus
brown_news_tagged = brown.tagged_words(categories='news', tagset='universal')
tag_fd = nltk.FreqDist(tag for (word, tag) in brown_news_tagged)

print("Suspect Roles (Tags) Frequency:")
tag_fd.tabulate()

## 2. Identity Verification (POS Tagging)

We need a "Scanner" to automatically assign tags to words.

### Method A: The Rules (Regex Tagger)
Quick and dirty. If it ends in "-ing", it's probably a Verb. If it ends in "-ly", it's an Adverb.

In [ ]:
patterns = [
    (r'.*ing$', 'VERB'),               # gerunds
    (r'.*ed$', 'VERB'),                # past tense
    (r'.*es$', 'VERB'),                # 3rd person singular present
    (r'.*ould$', 'VERB'),              # modals
    (r'.*\'s$', 'NOUN'),              # possessive nouns
    (r'.*s$', 'NOUN'),                 # plural nouns
    (r'^-?[0-9]+(.[0-9]+)?$', 'NUM'),  # cardinal numbers
    (r'.*', 'NOUN')                    # default (Innocent until proven guilty)
]

regex_tagger = nltk.RegexpTagger(patterns)
sentence = "The flying spaceship landed softly".split()
print(f"Scanner Output: {regex_tagger.tag(sentence)}")

### Method B: The Informants (N-gram Taggers)
We check our database. "What role does this word usually play?" (Unigram) or "What role usually follows this one?" (Bigram).

In [ ]:
# Train our informants on the Brown news corpus
train_data = brown.tagged_sents(categories='news')
test_data = brown.tagged_sents(categories='fiction')

unigram_tagger = nltk.UnigramTagger(train_data)
print(f"Unigram Accuracy: {unigram_tagger.accuracy(test_data):.2%}")

bigram_tagger = nltk.BigramTagger(train_data, backoff=unigram_tagger)
print(f"Bigram Accuracy: {bigram_tagger.accuracy(test_data):.2%}")

## 3. The Shadow Path (Hidden Markov Models)

Sometimes, simple lookup isn't enough. Example: "The **bank** is closed" (Noun) vs "I will **bank** the plane" (Verb).

We model this as a **Hidden Markov Model**.
*   **Hidden States ($Q$)**: The Tags (We can't see them directly).
*   **Observations ($O$)**: The Words (What we see).

The suspect moves based on two probabilities:
1.  **Transition ($A$)**: $P(t_i | t_{i-1})$ - Probability of a tag following another.
2.  **Emission ($B$)**: $P(w_i | t_i)$ - Probability of a tag generating a specific word.

## 4. The Optimal Path (Viterbi Algorithm)

To solve the case, we need to find the *single most likely sequence* of hidden tags. The "Greedy" approach (best tag at each step independently) fails. We need a global optimum.

### The Math Behind the Magic
We calculate a score $v_t(j)$ for every state $j$ at time $t$. This score represents the probability of the best path ending in state $j$ at time $t$.

$$ v_t(j) = \max_{i} ( v_{t-1}(i) \times P(State_j | State_i) \times P(Observation_t | State_j) ) $$

### Manual Detective Work (The Trace)
Let's trace our suspect manually before coding.
**Scenario**:
*   **States**: Rainy, Sunny
*   **Observations**: `walk`, `shop`, `clean`

**Step 1 ($t=0$): Observation 'walk'**
*   $P(Rainy) = Start(Rainy) \times Emit(walk|Rainy) = 0.6 \times 0.1 = 0.06$
*   $P(Sunny) = Start(Sunny) \times Emit(walk|Sunny) = 0.4 \times 0.6 = 0.24$ (Winner so far)

**Step 2 ($t=1$): Observation 'shop'**
We try to come from Rainy or Sunny:
*   Path to **Rainy**:
    *   From Rainy: $0.06 \times P(R|R) \times P(shop|R) = 0.06 \times 0.7 \times 0.4 = 0.0168$
    *   From Sunny: $0.24 \times P(R|S) \times P(shop|R) = 0.24 \times 0.4 \times 0.4 = 0.0384$ (Best path to Rainy comes from Sunny)
*   Path to **Sunny**:
    *   From Rainy: $0.06 \times P(S|R) \times P(shop|S) = ...$
    *   From Sunny: $0.24 \times P(S|S) \times P(shop|S) = ...$

**Agent IoB, build this Viterbi Decoder from scratch.**

In [ ]:
def viterbi(obs, states, start_p, trans_p, emit_p):
    V = [{}] # Trellis diagram
    path = {} # Backpointers
 
    # Initialize base cases (t=0)
    for y in states:
        V[0][y] = start_p[y] * emit_p[y].get(obs[0], 0.00001)
        path[y] = [y]
 
    # Run Viterbi for t > 0
    for t in range(1, len(obs)):
        V.append({})
        newpath = {}
 
        for y in states:
            # (prob, state) = max(prev_prob * trans_prob * emit_prob)
            (prob, state) = max(
                (V[t-1][y0] * trans_p[y0].get(y, 0) * emit_p[y].get(obs[t], 0.00001), y0) 
                for y0 in states
            )
            V[t][y] = prob
            newpath[y] = path[state] + [y]
 
        path = newpath
 
    n = len(obs) - 1
    (prob, state) = max((V[n][y], y) for y in states)
    return (prob, path[state])

# Mission Data
states = ('Rainy', 'Sunny')
observations = ('walk', 'shop', 'clean')
start_probability = {'Rainy': 0.6, 'Sunny': 0.4}
transition_probability = {
   'Rainy' : {'Rainy': 0.7, 'Sunny': 0.3},
   'Sunny' : {'Rainy': 0.4, 'Sunny': 0.6},
}
emission_probability = {
   'Rainy' : {'walk': 0.1, 'shop': 0.4, 'clean': 0.5},
   'Sunny' : {'walk': 0.6, 'shop': 0.3, 'clean': 0.1},
}

prob, result = viterbi(observations, states, start_probability, transition_probability, emission_probability)
print(f"Detected Path: {result}")
print(f"Confidence: {prob:.5f}")

## 5. Extracting Intel (Named Entity Recognition)

We have located the grammar, now we locate the targets: **PERSON**, **ORG**, **GPE** (Geopolitical Entity).

We use the **IOB** (Inside-Outside-Beginning) tagging scheme.
*   **B-ORG**: Beginning of an Organization.
*   **I-ORG**: Inside an Organization.
*   **O**: Outside any entity.

In [ ]:
nlp = spacy.load("en_core_web_sm")
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")

print(f"{'Text':<15} {'Label':<10} {'Description'}")
print("-"*40)
for ent in doc.ents:
    print(f"{ent.text:<15} {ent.label_:<10} {spacy.explain(ent.label_)}")

from spacy import displacy
displacy.render(doc, style="ent", jupyter=True)

## 6. Contextual Forensics (Conditional Random Fields)

HMMs look at the immediate past. **CRFs** look at the entire scene.
They are "Discriminative" models (modeling $P(Y|X)$ directly) rather than "Generative" (modeling $P(X|Y)$).

We define **Feature Functions** to verify suspects based on context:
*   "Is the word capitalized?"
*   "Is the previous word 'Mr.'?"
*   "Is the next word 'Street'?"

In [ ]:
# Example Feature Extraction for a CRF
def word2features(sent, i):
    word = sent[i][0]
    postag = sent[i][1]

    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],  # Suffix
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'postag': postag,
    }
    if i > 0:
        word1 = sent[i-1][0]
        postag1 = sent[i-1][1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
        })
    else:
        features['BOS'] = True # Beginning of Sentence

    return features

sent = [('Agent', 'NN'), ('IoB', 'NN'), ('solved', 'VBD'), ('it', 'PRP')]
print("Features for 'IoB':")
word2features(sent, 1)

## 7. Mission Report (Evaluation)

To evaluate our agents, we use **Precision** (How many selected were correct?) and **Recall** (How many correct ones were selected?).

$$ F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall} $$

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns

y_true = ['B-PER', 'I-PER', 'O', 'B-LOC', 'O']
y_pred = ['B-PER', 'O',     'O', 'B-LOC', 'O']

labels = sorted(list(set(y_true)))
cm = confusion_matrix(y_true, y_pred, labels=labels)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Mission Confusion Matrix')
plt.show()

print(f"Accuracy: {accuracy_score(y_true, y_pred):.2f}")